## **DualGate-DEC:** Dual-Graph Gated Representation with Calibrated Potts-Regularized Deep Embedding Clustering for Spatial Multi-Omics.

> **Hardware Acceleration**: In Google Colab, select **Runtime > Change runtime type > T4 GPU** before executing.


In [ ]:
# Cell 1: Global Configuration & Hyperparameter Initialization (DualGate-DEC)
import os

# 1. Dataset Selection: Choose indices to benchmark
# 0: Lymph_Node_A1, 1: Lymph_Node_D1, 2: Mouse_Brain_E11, 3: Mouse_Brain_E13, 4: Mouse_Brain_E15, 5: Mouse_Brain_E18
DATASET_INDICES = [0, 1, 2, 3, 4, 5]       # Choose datasets: [0], [0, 1], or all [0, 1, 2, 3, 4, 5]

# 2. Benchmark Seeds (Standard 9-seed protocol)
SEEDS = [42, 0, 7, 123, 1234, 2022, 2023, 2024, 1337]  # Benchmark seeds

# 3. Two-Stage Training Protocol (Calibrated 180-Epoch Protocol)
PRETRAIN_EPOCHS = 80                      # Stage 1: Representation pre-training (optimal knee of curve)
FINETUNE_EPOCHS = 100                     # Stage 2: Spatial Potts Consensus DEC fine-tuning

# 4. Clustering & Loss Hyperparameters
TOOL = 'kmeans'                            # Pretrain evaluation tool ('kmeans' or 'mclust')
LAMBDA_DEC = 1.0                           # Calibrated weight of DEC KL divergence loss
LAMBDA_SPATIAL = 0.15                      # Tempered spatial Potts consensus strength
LR = 1e-3                                  # Stage 1 learning rate (Stage 2 uses 1e-4)
P_UPDATE_INTERVAL = 5                      # Periodic target distribution freezing interval (T=5 epochs)

# Spatial Regularization Weight (γ)
GAMMA_SPATIAL = 10.0

#  Number of Neighbors (k)
NUM_NEIGHBORS = 15

# 5. Output Directory & Structured Subdirectories
OUTPUT_DIR = "DualGate_DEC_outputs"
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results")
VISUALS_DIR = os.path.join(OUTPUT_DIR, "visuals")
BIOMARKERS_DIR = os.path.join(OUTPUT_DIR, "biomarkers")

for d in [OUTPUT_DIR, RESULTS_DIR, VISUALS_DIR, BIOMARKERS_DIR]:
    os.makedirs(d, exist_ok=True)

print("="*75)
print(" DualGate-DEC: Global Configuration Initialized Successfully")
print("="*75)
print(f" * Datasets Selected : {DATASET_INDICES}")
print(f" * Seeds ({len(SEEDS)} Total)    : {SEEDS}")
print(f" * Epoch Schedule    : {PRETRAIN_EPOCHS} Pre-training + {FINETUNE_EPOCHS} DEC = {PRETRAIN_EPOCHS + FINETUNE_EPOCHS} Total Epochs")
print(f" * Calibrated Losses : lambda_DEC = {LAMBDA_DEC}, lambda_Spatial = {LAMBDA_SPATIAL}, Base LR = {LR}")
print(f" * Root Output Dir   : '{OUTPUT_DIR}/'")
print(f"   |-- Results Dir   : '{RESULTS_DIR}/' (CSVs & AnnData objects)")
print(f"   |-- Visuals Dir   : '{VISUALS_DIR}/' (Publication 300 DPI figures)")
print(f"   +-- Biomarkers Dir: '{BIOMARKERS_DIR}/' (Marker genes & pathways)")
print("="*75 + "\n")


In [ ]:
# Cell 2: Environment & Package Installation in Colab
!pip install -q scanpy anndata scikit-misc rpy2 "gdown>=5.1.0" torch_geometric
!R -e "if (!requireNamespace('mclust', quietly=TRUE)) install.packages('mclust', repos='https://cloud.r-project.org')"


In [ ]:
import time
# Cell 3: Imports & Environment Configuration
import os
import sys
import copy
import random
import numpy as np
import pandas as pd
import scipy
import scipy.sparse as sp
from typing import Optional, Tuple, Dict, List

import sklearn
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    adjusted_mutual_info_score,
    homogeneity_score,
    v_measure_score,
    fowlkes_mallows_score,
    silhouette_score,
    calinski_harabasz_score,
    davies_bouldin_score
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv

import scanpy as sc
import anndata as ad
import matplotlib.pyplot as plt
import seaborn as sns

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[Device] Using compute engine: {device}")
if torch.cuda.is_available():
    print(f"[GPU] Model: {torch.cuda.get_device_name(0)} | Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ['PYTHONHASHSEED'] = str(seed)

set_seed(42)

def init_r_mclust():
    try:
        import rpy2.robjects as robjects
        robjects.r['options'](warn=-1)
        robjects.r.library("mclust")
        return True
    except Exception as e:
        print(f"[Warning] R mclust unavailable ({e}). Fallback to KMeans.")
        return False



In [ ]:
# Cell 4: Universal Preprocessing Engine
def clr_normalize_each_cell(adata, inplace=True):
    def seurat_clr(x):
        s = np.sum(np.log1p(x[x > 0]))
        exp = np.exp(s / len(x)) if len(x) > 0 else 1.0
        return np.log1p(x / (exp + 1e-12))

    if not inplace:
        adata = adata.copy()

    X = adata.X.toarray() if sp.issparse(adata.X) else np.array(adata.X)
    adata.X = np.apply_along_axis(seurat_clr, 1, X)
    return adata

def tfidf(X):
    idf = X.shape[0] / (X.sum(axis=0) + 1e-12)
    if sp.issparse(X):
        tf = X.multiply(1.0 / (X.sum(axis=1) + 1e-12))
        return sp.csr_matrix(tf.multiply(idf))
    else:
        tf = X / (X.sum(axis=1, keepdims=True) + 1e-12)
        return tf * idf

def preprocess_universal(adata_RNA, adata_omics2, dataset_name, n_hvg=3000):
    sc.pp.filter_genes(adata_RNA, min_cells=10)
    try:
        sc.pp.highly_variable_genes(adata_RNA, flavor="seurat_v3", n_top_genes=n_hvg)
    except Exception:
        sc.pp.normalize_total(adata_RNA, target_sum=1e4)
        sc.pp.log1p(adata_RNA)
        sc.pp.highly_variable_genes(adata_RNA, flavor="seurat", n_top_genes=n_hvg)

    if 'log1p' not in adata_RNA.uns:
        sc.pp.normalize_total(adata_RNA, target_sum=1e4)
        sc.pp.log1p(adata_RNA)
    sc.pp.scale(adata_RNA)

    RNA_expression = adata_RNA[:, adata_RNA.var['highly_variable']].X
    if sp.issparse(RNA_expression):
        RNA_expression = RNA_expression.toarray()

    # Second Modality Preprocessing
    if dataset_name.startswith("10x"):
        # ADT Protein Modality
        adata_omics2 = clr_normalize_each_cell(adata_omics2)
        sc.pp.scale(adata_omics2)
        ADT_expression = adata_omics2.X.toarray() if sp.issparse(adata_omics2.X) else np.array(adata_omics2.X)
    else:
        # ATAC Epigenomic Modality
        adata_omics2.X = tfidf(adata_omics2.X)
        sc.pp.normalize_per_cell(adata_omics2, counts_per_cell_after=1e4)
        sc.pp.log1p(adata_omics2)
        n_comps = min(50, adata_omics2.shape[1])
        pca_model = PCA(n_components=n_comps, random_state=42)
        X_atac = adata_omics2.X.toarray() if sp.issparse(adata_omics2.X) else adata_omics2.X
        ADT_expression = pca_model.fit_transform(X_atac)

    return RNA_expression, ADT_expression, adata_RNA, adata_omics2


In [ ]:
# Cell 5: Multi-View Consensus Graph & Spatial Adjacency Construction
class DualGraphData(Data):
    def __init__(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                 dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        super().__init__()
        self.x_RNA = x_RNA
        self.x_ADT = x_ADT
        self.sim_edge_index = sim_edge_index
        self.sim_edge_weight = sim_edge_weight
        self.dist_edge_index = dist_edge_index
        self.dist_edge_weight = dist_edge_weight
        self.common_edge_index = common_edge_index
        self.common_edge_weight = common_edge_weight

def build_dual_graph(RNA_expression, ADT_expression, cell_positions, device='cpu', num_neighbors=15):
    # 1. Cosine Feature Similarity KNN Graph
    similarity_matrix = cosine_similarity(RNA_expression)
    nbrs = NearestNeighbors(n_neighbors=num_neighbors + 1, metric='cosine').fit(RNA_expression)
    _, indices = nbrs.kneighbors(RNA_expression)

    adjacency_matrix = np.zeros_like(similarity_matrix, dtype=int)
    for i in range(len(RNA_expression)):
        for j in indices[i][1:]:
            adjacency_matrix[i, j] = 1
            adjacency_matrix[j, i] = 1

    sim_edge_index = torch.tensor(np.array(np.nonzero(adjacency_matrix)), dtype=torch.long).to(device)
    sim_edge_weight = torch.tensor(similarity_matrix[adjacency_matrix > 0], dtype=torch.float).to(device)

    # 2. Spatial Distance KNN Graph
    knn_graph = kneighbors_graph(cell_positions, n_neighbors=num_neighbors, mode='distance', include_self=False)
    knn_graph = knn_graph.maximum(knn_graph.T)
    dist_edge_index = torch.tensor(np.array(knn_graph.nonzero()), dtype=torch.long).to(device)
    dist_edge_weight = torch.tensor(knn_graph.data, dtype=torch.float).to(device)

    # 3. Common Edge Pure Intersection (Consensus)
    sim_edges = set(zip(sim_edge_index[0].tolist(), sim_edge_index[1].tolist()))
    dist_edges = set(zip(dist_edge_index[0].tolist(), dist_edge_index[1].tolist()))
    common_edges = sim_edges.intersection(dist_edges)

    common_edge_index = torch.tensor(list(zip(*common_edges)), dtype=torch.long).to(device)
    common_edge_weight = torch.ones(common_edge_index.size(1), dtype=torch.float).to(device)

    x_RNA = torch.tensor(RNA_expression, dtype=torch.float).to(device)
    x_ADT = torch.tensor(ADT_expression, dtype=torch.float).to(device)

    return DualGraphData(
        x_RNA, x_ADT,
        sim_edge_index, sim_edge_weight,
        dist_edge_index, dist_edge_weight,
        common_edge_index, common_edge_weight
    )

def compute_spatial_adj_norm(dist_edge_index, num_nodes, device='cpu'):
    """Construct row-normalized spatial adjacency matrix for Potts MRF consensus."""
    adj = torch.sparse_coo_tensor(dist_edge_index, torch.ones(dist_edge_index.size(1), device=device), size=(num_nodes, num_nodes)).to_dense()
    adj = adj + torch.eye(num_nodes, device=device)
    deg = adj.sum(dim=1, keepdim=True)
    adj_norm = adj / (deg + 1e-12)
    return adj_norm


In [ ]:
# Cell 6: Neural Architecture & Gated Attention Fusion (DualGate-DEC) Modules
class GatedAttentionFusion(nn.Module):
    """
    Spot-Adaptive Gated Attention Fusion Module.
    Computes a learned, per-spot, feature-wise attention gate g in (0, 1)^d:
        g = Sigmoid(W_g [z1 || z2] + b_g)
        z_fused = g ⊙ z1 + (1 - g) ⊙ z2
    Followed by an output projection layer to refine the fused manifold.
    """
    def __init__(self, dim: int = 64):
        super().__init__()
        self.gate_net = nn.Sequential(
            nn.Linear(2 * dim, dim),
            nn.Sigmoid()
        )
        self.out_proj = nn.Linear(dim, dim)

    def forward(self, z1: torch.Tensor, z2: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        combined = torch.cat([z1, z2], dim=1)
        gate = self.gate_net(combined)
        fused = gate * z1 + (1.0 - gate) * z2
        fused = self.out_proj(fused)
        return fused, gate

class SpatialPottsDEC(nn.Module):
    """
    Spatial Potts Markov Random Field Consensus Deep Embedding Clustering (DEC).
    Smooths target distribution P using spatial neighborhood consensus to suppress salt-and-pepper noise
    while preserving transcriptomic cluster boundaries.
    """
    def __init__(self, num_clusters: int, latent_dim: int = 64, alpha: float = 1.0, lambda_spatial: float = 0.15):
        super().__init__()
        self.num_clusters = num_clusters
        self.latent_dim = latent_dim
        self.alpha = alpha
        self.lambda_spatial = lambda_spatial
        self.cluster_centers = nn.Parameter(torch.Tensor(num_clusters, latent_dim))
        nn.init.xavier_uniform_(self.cluster_centers)

    def compute_q(self, z: torch.Tensor) -> torch.Tensor:
        # Student's t-distribution soft assignment
        dist = torch.sum((z.unsqueeze(1) - self.cluster_centers.unsqueeze(0)) ** 2, dim=2)
        q = 1.0 / (1.0 + dist / self.alpha)
        q = q ** ((self.alpha + 1.0) / 2.0)
        q = q / (torch.sum(q, dim=1, keepdim=True) + 1e-12)
        return q

    def compute_spatial_target_p(self, q: torch.Tensor, spatial_adj_norm: torch.Tensor) -> torch.Tensor:
        # Standard DEC target distribution
        weight = q ** 2 / (torch.sum(q, dim=0, keepdim=True) + 1e-12)
        p_dec = weight / (torch.sum(weight, dim=1, keepdim=True) + 1e-12)

        # Spatial Potts neighborhood consensus
        spatial_consensus = torch.mm(spatial_adj_norm, q)

        # Potts modulated target
        p_spatial = p_dec * torch.exp(self.lambda_spatial * spatial_consensus)
        p_final = p_spatial / (torch.sum(p_spatial, dim=1, keepdim=True) + 1e-12)
        return p_final

    def forward(self, z: torch.Tensor, p_target: Optional[torch.Tensor] = None):
        q = self.compute_q(z)
        if p_target is not None:
            kl_loss = F.kl_div(q.log(), p_target, reduction='batchmean')
        else:
            kl_loss = torch.tensor(0.0, device=z.device)
        return q, kl_loss

class DualGCN_Gated(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, q_dim, dropout=0.0):
        super(DualGCN_Gated, self).__init__()
        # RNA dual branches (similarity & distance)
        self.x_RNA1 = GCNConv(in_channels, hidden_channels)
        self.x_RNA2 = GCNConv(in_channels, hidden_channels)
        self.protein3 = GCNConv(q_dim, out_channels)

        self.sim_conv = GCNConv(hidden_channels, out_channels)
        self.dist_conv = GCNConv(hidden_channels, out_channels)

        # Spot-Adaptive Gated Attention Fusion Modules
        # Fusion 1: Within-Modality (RNA Sim vs. RNA Dist)
        self.fusion_layer1 = GatedAttentionFusion(dim=out_channels)
        # Fusion 2: Between-Modality (Fused RNA vs. Auxiliary ADT/ATAC)
        self.fusion_layer2 = GatedAttentionFusion(dim=out_channels)

        # Decoupled Multi-Task Decoders
        self.dropout = dropout
        self.deconv1 = nn.Linear(out_channels, hidden_channels)
        self.deconv2 = nn.Linear(hidden_channels, in_channels)
        self.deconv4 = nn.Linear(hidden_channels, q_dim)
        self.deconv5 = nn.Linear(hidden_channels, in_channels + q_dim)

    def forward(self, x_RNA, x_ADT, sim_edge_index, sim_edge_weight,
                dist_edge_index, dist_edge_weight, common_edge_index, common_edge_weight):
        xs = F.relu(self.x_RNA1(x_RNA, sim_edge_index, sim_edge_weight))
        xs = F.dropout(xs, self.dropout, training=self.training)

        xd = F.relu(self.x_RNA2(x_RNA, dist_edge_index, dist_edge_weight))
        xd = F.dropout(xd, self.dropout, training=self.training)

        x_sim = self.sim_conv(xs, sim_edge_index, sim_edge_weight)
        x_dist = self.dist_conv(xd, dist_edge_index, dist_edge_weight)
        pro = self.protein3(x_ADT, common_edge_index, common_edge_weight)

        # Dynamic Gated Fusion
        fused, gate_spatial = self.fusion_layer1(x_sim, x_dist)
        fused_pro, gate_modality = self.fusion_layer2(fused, pro)
        return x_sim, x_dist, fused, fused_pro, pro, gate_spatial, gate_modality

    def reconstruct_joint(self, z):
        return self.deconv5(F.relu(self.deconv1(z)))

    def reconstruct_rna(self, z):
        return self.deconv2(F.relu(self.deconv1(z)))

    def reconstruct_aux(self, z):
        return self.deconv4(F.relu(self.deconv1(z)))

class DualGate_DEC(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels, q_dim, num_clusters,
                 beta=25.0, gamma=10.0, delta=1.0, lambda_dec=1.0, lambda_spatial=0.15, dropout=0.0):
        super().__init__()
        self.gcn = DualGCN_Gated(in_channels, hidden_channels, out_channels, q_dim, dropout=dropout)
        self.dec = SpatialPottsDEC(num_clusters=num_clusters, latent_dim=out_channels, lambda_spatial=lambda_spatial)
        self.beta = beta
        self.gamma = gamma
        self.delta = delta
        self.lambda_dec = lambda_dec
        self.l1_lambda = 1e-4
        self.l2_lambda = 1e-3

    def init_cluster_centers(self, centers_numpy):
        """Initialize DEC prototype centers using KMeans in aligned latent coordinates."""
        self.dec.cluster_centers.data.copy_(torch.tensor(centers_numpy, dtype=torch.float32, device=self.dec.cluster_centers.device))

    def forward(self, data, compute_dec=False, p_target=None):
        x_sim, x_dist, fused, fused_pro, pro, gate_sp, gate_mod = self.gcn(
            data.x_RNA, data.x_ADT,
            data.sim_edge_index, data.sim_edge_weight,
            data.dist_edge_index, data.dist_edge_weight,
            data.common_edge_index, data.common_edge_weight
        )
        if compute_dec:
            q, kl_loss = self.dec(fused_pro, p_target)
            return x_sim, x_dist, fused, fused_pro, pro, q, kl_loss, gate_sp, gate_mod
        return x_sim, x_dist, fused, fused_pro, pro, gate_sp, gate_mod

    def cosine_similarity(self, emb):
        mat = torch.matmul(emb, emb.T)
        norm = torch.norm(emb, p=2, dim=1).reshape((emb.shape[0], 1))
        mat = torch.div(mat, torch.matmul(norm, norm.T) + 1e-12)
        mat = torch.where(torch.isnan(mat), torch.zeros_like(mat), mat)
        mat = mat - torch.diag_embed(torch.diag(mat))
        return mat

    def spatial_regularization_loss(self, emb, dist_edge_index, dist_edge_weight):
        num_nodes = emb.size(0)
        graph_nei = torch.sparse_coo_tensor(
            dist_edge_index,
            torch.ones_like(dist_edge_weight),
            size=(num_nodes, num_nodes)
        ).to_dense()
        graph_neg = 1.0 - graph_nei
        sim_mat = torch.sigmoid(self.cosine_similarity(emb))
        neigh_loss = torch.mul(graph_nei, torch.log(sim_mat + 1e-10)).mean()
        neg_loss = torch.mul(graph_neg, torch.log(1.0 - sim_mat + 1e-10)).mean()
        return -(neigh_loss + neg_loss) / 2.0

    def compute_losses(self, data, sim_z, dist_z, fused_z, fused_pro, pro, kl_loss=None, stage=1):
        combined_raw = torch.cat([data.x_RNA, data.x_ADT], dim=1)
        l_rec = F.mse_loss(combined_raw, self.gcn.reconstruct_joint(fused_pro))
        l_sim = F.mse_loss(data.x_RNA, self.gcn.reconstruct_rna(sim_z))
        l_dist = F.mse_loss(data.x_RNA, self.gcn.reconstruct_rna(dist_z))
        l_aux = F.mse_loss(data.x_ADT, self.gcn.reconstruct_aux(pro))

        l_spatial = self.spatial_regularization_loss(fused_z, data.dist_edge_index, data.dist_edge_weight)

        l1_reg = sum(torch.sum(torch.abs(p)) for p in self.parameters())
        l2_reg = sum(torch.sum(p ** 2) for p in self.parameters())
        reg_loss = self.l1_lambda * l1_reg + self.l2_lambda * l2_reg

        # In Stage 2, moderate reconstruction weight so DEC KL divergence actively sharpens clusters
        beta_curr = self.beta if stage == 1 else (self.beta * 0.2)
        rec_term = beta_curr * (l_rec + l_sim + l_dist + l_aux)
        spatial_term = self.gamma * l_spatial
        reg_term = self.delta * reg_loss

        total_loss = rec_term + spatial_term + reg_term
        if stage == 2 and kl_loss is not None:
            total_loss = total_loss + self.lambda_dec * kl_loss

        return total_loss, l_rec, l_spatial, (kl_loss.item() if kl_loss is not None else 0.0)


In [ ]:
# Cell 7: Clustering & Two-Stage Training Engine (DualGate-DEC): DualGate-DEC (180 Epochs)
def mclust_R(adata_or_embeddings, num_cluster, modelNames='EEE', use_pca=True, n_comps=20, random_seed=42):
    np.random.seed(random_seed)
    if init_r_mclust():
        try:
            import rpy2.robjects as robjects
            from rpy2.robjects import pandas2ri, default_converter
            from rpy2.robjects.conversion import localconverter
            robjects.r.library("mclust")
            robjects.r["set.seed"](random_seed)
            rmclust = robjects.r["Mclust"]

            X = np.array(adata_or_embeddings, dtype=np.float64)
            if use_pca and X.shape[1] > n_comps:
                comps = min(n_comps, X.shape[1], X.shape[0] - 1)
                pca_model = PCA(n_components=comps, random_state=random_seed)
                X = pca_model.fit_transform(X)

            df = pd.DataFrame(X, columns=[f"PC{i+1}" for i in range(X.shape[1])])
            subset_size = min(300, X.shape[0])
            subset_indices = robjects.IntVector(list(np.random.choice(range(1, X.shape[0] + 1), subset_size, replace=False)))
            init_list = robjects.ListVector({'subset': subset_indices})

            res = None
            with localconverter(default_converter + pandas2ri.converter):
                try:
                    res = rmclust(df, G=num_cluster, modelNames=modelNames, initialization=init_list)
                except Exception:
                    res = rmclust(df, G=num_cluster)

            if res is not None and hasattr(res, 'names') and 'classification' in res.names:
                return np.array(res['classification']).astype(int)
        except Exception as e:
            print(f"[Notice] mclust fallback ({e}). Using KMeans.")
    from sklearn.cluster import KMeans
    return KMeans(n_clusters=num_cluster, n_init=10, random_state=random_seed).fit_predict(adata_or_embeddings)

def cluster_embeddings(embeddings, num_clusters, method='kmeans', random_seed=42):
    if method == 'mclust':
        return mclust_R(embeddings, num_cluster=num_clusters, random_seed=random_seed)
    from sklearn.cluster import KMeans
    return KMeans(n_clusters=num_clusters, n_init=10, random_state=random_seed).fit_predict(embeddings)

def train_DualGate_DEC(
    model: DualGate_DEC,
    data: DualGraphData,
    spatial_adj_norm: torch.Tensor,
    pretrain_epochs: int = 250,
    finetune_epochs: int = 150,
    lr: float = 1e-3,
    num_clusters: int = 10,
    tool: str = 'kmeans',
    seed: int = 42,
    p_update_interval: int = 5
):
    """
    Calibrated Two-Stage Training Protocol with Gated Attention Fusion: DualGate-DEC (180 Total Epochs: 80 Pre-train + 100 Potts DEC):
      - Stage 1 (Epochs 1-250): Multimodal Representation Pre-training with Gated Fusion
      - Bridge Transition: Restores model weights to peak Stage-1 state_dict and seeds DEC prototypes
      - Stage 2 (Epochs 251-400): Spatial Potts Consensus DEC Fine-Tuning with periodic target freezing
      - Returns overall best checkpoint across ALL 400 epochs along with learned attention gates.
    """
    optimizer_stage1 = torch.optim.Adam(model.parameters(), lr=lr)

    best_sil = -1.0
    best_epoch = -1
    best_stage = "Pre-train"
    best_embeddings = None
    best_labels = None
    best_imputed_rna = None
    best_gates = None
    best_model_state_overall = None

    best_sil_stage1 = -1.0
    best_epoch_stage1 = -1
    best_model_state_stage1 = None
    best_embeddings_stage1 = None

    history = {
        'total_loss': [],
        'recon_loss': [],
        'spatial_loss': [],
        'kl_loss': [],
        'silhouette': []
    }

    # =========================================================================
    # STAGE 1: Representation Pre-training (Epochs 1 to 250)
    # =========================================================================
    print(f"\n{'='*75}\n[Stage 1/2] Gated Fusion Pre-training ({pretrain_epochs} Epochs | {tool})\n{'='*75}")
    for epoch in range(pretrain_epochs):
        model.train()
        optimizer_stage1.zero_grad()
        sim_z, dist_z, fused_z, fused_pro, pro, g_sp, g_mod = model(data, compute_dec=False)
        loss, l_rec, l_sp, _ = model.compute_losses(data, sim_z, dist_z, fused_z, fused_pro, pro, stage=1)
        loss.backward()
        optimizer_stage1.step()

        # Evaluation mode
        model.eval()
        with torch.no_grad():
            sim_z_eval, dist_z_eval, fused_z_eval, fused_pro_eval, pro_eval, g_sp_eval, g_mod_eval = model(data, compute_dec=False)
            epoch_emb = fused_pro_eval.cpu().numpy()

        epoch_labels = cluster_embeddings(epoch_emb, num_clusters, method=tool, random_seed=seed)
        try:
            sil = float(silhouette_score(epoch_emb, epoch_labels))
        except Exception:
            sil = -1.0

        history['total_loss'].append(loss.item())
        history['recon_loss'].append(l_rec.item())
        history['spatial_loss'].append(l_sp.item())
        history['kl_loss'].append(0.0)
        history['silhouette'].append(sil)

        if sil > best_sil:
            best_sil = sil
            best_epoch = epoch + 1
            best_stage = "Pre-train"
            best_embeddings = epoch_emb.copy()
            best_labels = epoch_labels.copy()
            best_gates = (g_sp_eval.cpu().numpy(), g_mod_eval.cpu().numpy())
            best_model_state_overall = copy.deepcopy(model.state_dict())
            with torch.no_grad():
                best_imputed_rna = model.gcn.reconstruct_rna(sim_z_eval).cpu().numpy()

        if sil > best_sil_stage1:
            best_sil_stage1 = sil
            best_epoch_stage1 = epoch + 1
            best_model_state_stage1 = copy.deepcopy(model.state_dict())
            best_embeddings_stage1 = epoch_emb.copy()

        if (epoch + 1) % 10 == 0:
            print(f"Pretrain Ep {epoch+1:3d}/{pretrain_epochs} | Loss: {loss.item():.4f} | Recon: {l_rec.item():.4f} | Sil: {sil:.4f} (Best: {best_sil:.4f} @ Ep {best_epoch})")

    # =========================================================================
    # BRIDGE: Restore Peak Stage-1 Weights & Seed Aligned DEC Prototypes
    # =========================================================================
    print(f"\n[Bridge Transition] Restoring peak Stage-1 model weights (Sil = {best_sil_stage1:.4f} @ Ep {best_epoch_stage1})...")
    if best_model_state_stage1 is not None:
        model.load_state_dict(best_model_state_stage1)

    model.eval()
    with torch.no_grad():
        _, _, _, fused_pro_best, _, _, _ = model(data, compute_dec=False)
        best_emb_aligned = fused_pro_best.cpu().numpy()

    kmeans_init = KMeans(n_clusters=num_clusters, n_init=20, random_state=seed)
    kmeans_init.fit(best_emb_aligned)
    model.init_cluster_centers(kmeans_init.cluster_centers_)
    print("DEC Cluster Prototypes successfully initialized in aligned latent space!")

    # =========================================================================
    # STAGE 2: Spatial Potts Consensus DEC Fine-Tuning (Epochs 251 to 400)
    # =========================================================================
    lr_stage2 = lr * 0.1  # Dedicated gentle fine-tuning rate (1e-4)
    optimizer_stage2 = torch.optim.Adam(model.parameters(), lr=lr_stage2)
    p_target = None

    print(f"\n{'='*75}\n[Stage 2/2] Gated Spatial Potts Consensus DEC Fine-Tuning ({finetune_epochs} Epochs | lr={lr_stage2:.1e})\n{'='*75}")
    for epoch in range(finetune_epochs):
        curr_epoch = pretrain_epochs + epoch + 1
        model.train()

        # Periodic target distribution update (T=5)
        if epoch % p_update_interval == 0 or p_target is None:
            with torch.no_grad():
                _, _, _, f_pro_curr, _, _, _ = model(data, compute_dec=False)
                q_curr = model.dec.compute_q(f_pro_curr)
                p_target = model.dec.compute_spatial_target_p(q_curr, spatial_adj_norm)

        optimizer_stage2.zero_grad()
        sim_z, dist_z, fused_z, fused_pro, pro, q, kl_loss, g_sp, g_mod = model(data, compute_dec=True, p_target=p_target)
        loss, l_rec, l_sp, kl_val = model.compute_losses(data, sim_z, dist_z, fused_z, fused_pro, pro, kl_loss=kl_loss, stage=2)
        loss.backward()
        optimizer_stage2.step()

        # Direct GPU DEC Evaluation
        model.eval()
        with torch.no_grad():
            sim_z_eval, dist_z_eval, fused_z_eval, fused_pro_eval, pro_eval, g_sp_eval, g_mod_eval = model(data, compute_dec=False)
            q_eval = model.dec.compute_q(fused_pro_eval)
            epoch_emb = fused_pro_eval.cpu().numpy()
            preds_q = q_eval.argmax(dim=1).cpu().numpy()

        n_unique_q = len(np.unique(preds_q))
        if n_unique_q >= 2:
            try:
                sil_q = float(silhouette_score(epoch_emb, preds_q))
            except Exception:
                sil_q = -1.0
        else:
            sil_q = -1.0

        try:
            km_preds = KMeans(n_clusters=num_clusters, n_init=5, random_state=seed).fit_predict(epoch_emb)
            sil_km = float(silhouette_score(epoch_emb, km_preds))
        except Exception:
            sil_km = -1.0

        # Pick superior clustering partition
        if sil_q >= sil_km and n_unique_q >= 2:
            sil = sil_q
            epoch_labels = preds_q
        else:
            sil = sil_km
            epoch_labels = km_preds

        history['total_loss'].append(loss.item())
        history['recon_loss'].append(l_rec.item())
        history['spatial_loss'].append(l_sp.item())
        history['kl_loss'].append(kl_val)
        history['silhouette'].append(sil)

        if sil > best_sil:
            best_sil = sil
            best_epoch = curr_epoch
            best_stage = "DEC-Potts"
            best_embeddings = epoch_emb.copy()
            best_labels = epoch_labels.copy()
            best_gates = (g_sp_eval.cpu().numpy(), g_mod_eval.cpu().numpy())
            best_model_state_overall = copy.deepcopy(model.state_dict())
            with torch.no_grad():
                best_imputed_rna = model.gcn.reconstruct_rna(sim_z_eval).cpu().numpy()

        if (epoch + 1) % 10 == 0:
            print(f"DEC Ep {epoch+1:3d}/{finetune_epochs} (Total {curr_epoch:3d}) | Loss: {loss.item():.4f} | KL: {kl_val:.4f} | Sil: {sil:.4f} (Peak: {best_sil:.4f} [{best_stage}] @ Ep {best_epoch})")

    if best_model_state_overall is not None:
        model.load_state_dict(best_model_state_overall)

    print(f"\n{'='*75}\n[Best Overall Checkpoint] Highest Silhouette = {best_sil:.4f} achieved in [{best_stage}] at Epoch {best_epoch}.\n{'='*75}")
    return best_embeddings, best_labels, best_imputed_rna, best_gates, best_epoch, best_stage, history


In [ ]:
# Cell 8: Dataset Directory & Automatic Download Resolver
CHOICES = [
    ("10x_human_lymph_node_A1", "https://drive.google.com/drive/folders/10z1N4MwW8Y49o8GlkYGBKVx1N7fiMuyC"),
    ("10x_human_lymph_node_D1", "https://drive.google.com/drive/folders/1-g_Ca2XMaMXF-MisuVY-wobWDX86O6zz"),
    ("Mouse_Brain_E11_S1", "https://drive.google.com/drive/folders/1zRwDJrYnks0LRzlAVRqPU7jE_OcStgPo"),
    ("Mouse_Brain_E13_S1", "https://drive.google.com/drive/folders/1GOufwIRjjfcd9Bi2GKtebzKoPCg2jVud"),
    ("Mouse_Brain_E15_S1", "https://drive.google.com/drive/folders/1rHkTL5OF5qPsEERypRGMS51SjUQ69tdD"),
    ("Mouse_Brain_E18_S1", "https://drive.google.com/drive/folders/1Xj1LNIAY93biS6JIMKNRODn5GvtCKADB"),
]

def resolve_dataset_files(dataset_name: str, folder_url: str):
    candidate_dirs = [
        f"data/6_Datasets/{dataset_name}",
        f"../data/6_Datasets/{dataset_name}",
        f"data/{dataset_name}",
        f"/content/data/{dataset_name}",
        f"/content/drive/MyDrive/Colab/data/{dataset_name}",
        f"/content/drive/MyDrive/Colab/ARISE/data/{dataset_name}",
        f"/content/drive/MyDrive/Colab/{dataset_name}"
    ]
    base_dir = None
    for c_dir in candidate_dirs:
        if os.path.exists(c_dir):
            base_dir = c_dir
            break

    if base_dir is None:
        base_dir = f"data/{dataset_name}"
        os.makedirs(base_dir, exist_ok=True)
        print(f"Downloading dataset {dataset_name} using gdown...")
        os.system(f'gdown --folder "{folder_url}" --output "{base_dir}"')

    rna_path = os.path.join(base_dir, "adata_RNA.h5ad")
    if dataset_name.startswith("10x"):
        aux_path = os.path.join(base_dir, "adata_ADT.h5ad")
        anno_path = os.path.join(base_dir, "annotation.csv")
        gt_col = "manual-anno"
    else:
        aux_path = os.path.join(base_dir, "adata_ATAC.h5ad")
        anno_path = os.path.join(base_dir, "anno.csv")
        gt_col = "cluster"

    if not (os.path.exists(rna_path) and os.path.exists(aux_path) and os.path.exists(anno_path)):
        print(f"Re-downloading {dataset_name}...")
        os.system(f'gdown --folder "{folder_url}" --output "{base_dir}"')

    return rna_path, aux_path, anno_path, gt_col


In [ ]:
# Cell 9: Main Execution Loop across Datasets & Seeds (DualGate-DEC)

RESULTS_DIR = os.path.join(OUTPUT_DIR, 'results') if 'OUTPUT_DIR' in locals() else 'DualGate_DEC_outputs/results'
os.makedirs(RESULTS_DIR, exist_ok=True)

all_results = []
all_histories = {}
all_dataset_adata = {}

for d_idx in DATASET_INDICES:
    dataset_name, folder_url = CHOICES[d_idx]
    print(f"\n{'#'*80}\n STARTING DATASET: {dataset_name} (DualGate-DEC)\n{'#'*80}")

    rna_path, aux_path, anno_path, gt_col = resolve_dataset_files(dataset_name, folder_url)
    adata_RNA = sc.read_h5ad(rna_path)
    adata_aux = sc.read_h5ad(aux_path)
    anno_df = pd.read_csv(anno_path, index_col=0)

    # Safe ground truth extraction
    gt_values = anno_df[gt_col] if isinstance(anno_df[gt_col], pd.Series) else anno_df[gt_col].iloc[:, 0]
    adata_RNA.obs['ground_truth'] = gt_values.values
    adata_aux.obs['ground_truth'] = gt_values.values

    RNA_data, ADT_data, adata_RNA, adata_aux = preprocess_universal(adata_RNA, adata_aux, dataset_name)
    cell_positions = adata_RNA.obsm['spatial']
    graph_data = build_dual_graph(RNA_data, ADT_data, cell_positions, device=device, num_neighbors=NUM_NEIGHBORS)

    spatial_adj_norm = compute_spatial_adj_norm(graph_data.dist_edge_index, RNA_data.shape[0], device=device)
    n_clusters = int(adata_RNA.obs['ground_truth'].dropna().nunique())
    print(f"[+] Ground Truth Spatial Domains: {n_clusters} clusters")

    # Track best-performing seed representation for spatial visualization & AnnData export
    best_dataset_ari = -1.0
    top_emb = None
    top_labels = None
    top_imputed = None
    top_gates = None
    top_seed = SEEDS[0]

    for seed in SEEDS:
        print(f"\n{'-'*65}\n Dataset: {dataset_name} | Seed: {seed} | DualGate-DEC (180 Ep)\n{'-'*65}")
        set_seed(seed)

        model = DualGate_DEC(
            in_channels=RNA_data.shape[1],
            hidden_channels=512,
            out_channels=64,
            q_dim=ADT_data.shape[1],
            num_clusters=n_clusters,
            beta=25.0,
            gamma=GAMMA_SPATIAL,
            delta=1.0,
            lambda_dec=LAMBDA_DEC,
            lambda_spatial=LAMBDA_SPATIAL
        ).to(device)

        start_train_time = time.time()
        best_emb, best_labels, best_imputed_rna, best_gates, best_ep, best_stg, hist = train_DualGate_DEC(
            model=model,
            data=graph_data,
            spatial_adj_norm=spatial_adj_norm,
            pretrain_epochs=PRETRAIN_EPOCHS,
            finetune_epochs=FINETUNE_EPOCHS,
            lr=LR,
            num_clusters=n_clusters,
            tool=TOOL,
            seed=seed,
            p_update_interval=P_UPDATE_INTERVAL
        )
        train_duration = time.time() - start_train_time

        y_true = adata_RNA.obs['ground_truth'].astype(str)
        y_pred = best_labels.astype(str)

        ari = float(adjusted_rand_score(y_true, y_pred))
        nmi = float(normalized_mutual_info_score(y_true, y_pred))
        ami = float(adjusted_mutual_info_score(y_true, y_pred))
        homo = float(homogeneity_score(y_true, y_pred))
        v_meas = float(v_measure_score(y_true, y_pred))
        fmi = float(fowlkes_mallows_score(y_true, y_pred))
        sil = float(silhouette_score(best_emb, best_labels))
        chi = float(calinski_harabasz_score(best_emb, best_labels))
        dbi = float(davies_bouldin_score(best_emb, best_labels))

        # Check if current seed achieves highest ARI for spatial export
        if ari > best_dataset_ari or top_emb is None:
            best_dataset_ari = ari
            top_seed = seed
            top_emb = best_emb
            top_labels = best_labels
            top_imputed = best_imputed_rna
            top_gates = best_gates

        print(f"\n[Final Result] {dataset_name} | Seed {seed} -> ARI: {ari:.4f} | NMI: {nmi:.4f} | AMI: {ami:.4f} | Sil: {sil:.4f} | Best Ep: {best_ep} [{best_stg}]")

        if dataset_name not in all_histories:
            all_histories[dataset_name] = {}
        all_histories[dataset_name][seed] = hist

        all_results.append({
            'dataset': dataset_name,
            'seed': seed,
            'tool': TOOL,
            'ARI': ari,
            'NMI': nmi,
            'AMI': ami,
            'Homogeneity': homo,
            'V-Measure': v_meas,
            'FMI': fmi,
            'Silhouette': sil,
            'CHI': chi,
            'DBI': dbi,
            'best_epoch': best_ep,
            'best_stage': best_stg,
            'Execution_Time_s': round(train_duration, 2)
        })

    # Dataset Summary across all seeds: Mean ± Standard Deviation
    dataset_rows = [r for r in all_results if r['dataset'] == dataset_name]
    df_d = pd.DataFrame(dataset_rows)
    metrics_to_report = ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-Measure', 'FMI', 'Silhouette', 'CHI', 'DBI']

    print(f"\n{'='*80}")
    print(f" DATASET BENCHMARK SUMMARY (Mean ± Std over {len(SEEDS)} Seeds): {dataset_name} (DualGate-DEC)")
    print(f"{'='*80}")
    for m in metrics_to_report:
        if m in df_d.columns:
            m_mean = df_d[m].mean()
            m_std = df_d[m].std()
            print(f"  • {m:<14}: {m_mean:8.4f} ± {m_std:7.4f}")
    if 'Execution_Time_s' in df_d.columns:
        dur_mean = df_d['Execution_Time_s'].mean()
        dur_std = df_d['Execution_Time_s'].std()
        dur_total = df_d['Execution_Time_s'].sum()
        dur_mins, dur_secs = divmod(int(dur_mean), 60)
        tot_mins, tot_secs = divmod(int(dur_total), 60)
        print(f"  • {'Execution Time':<14}: {dur_mean:8.2f}s ± {dur_std:6.2f}s per seed (~{dur_mins}m {dur_secs:02d}s) | Total: {tot_mins}m {tot_secs:02d}s")
    print(f"{'='*80}\n")

    # Announce and save AnnData outputs for top-performing seed (Peak ARI)
    print(f"\n[Best ARI Seed Selected] {dataset_name} -> Seed {top_seed} achieved peak ARI ({best_dataset_ari:.4f}) and is preserved for spatial visualization and biomarker discovery.")
    adata_hvg = adata_RNA[:, adata_RNA.var['highly_variable']].copy()
    adata_hvg.obsm['DualGate_DEC_emb'] = top_emb
    adata_hvg.obs['DualGate_DEC_domain'] = pd.Categorical(top_labels.astype(str))
    adata_hvg.uns['selected_seed'] = int(top_seed)
    adata_hvg.uns['selected_ari'] = float(best_dataset_ari)
    if top_imputed is not None:
        adata_hvg.layers['imputed'] = top_imputed
    if top_gates is not None:
        g_sp, g_mod = top_gates
        adata_hvg.obsm['gate_spatial'] = g_sp
        adata_hvg.obsm['gate_modality'] = g_mod
        adata_hvg.obs['mean_rna_attention'] = g_mod.mean(axis=1)

    adata_hvg.write_h5ad(os.path.join(RESULTS_DIR, f"DualGate_DEC_adata_{dataset_name}.h5ad"))
    all_dataset_adata[dataset_name] = adata_hvg.copy()

    # Update parent adata_RNA
    adata_RNA.obsm['DualGate_DEC_emb'] = top_emb
    adata_RNA.obs['DualGate_DEC_domain'] = pd.Categorical(top_labels.astype(str))
    adata_RNA.uns['selected_seed'] = int(top_seed)
    if top_imputed is not None:
        imputed_full = (adata_RNA.X.toarray() if sp.issparse(adata_RNA.X) else np.array(adata_RNA.X, dtype=np.float32)).copy()
        hvg_mask = adata_RNA.var['highly_variable'].values
        imputed_full[:, hvg_mask] = top_imputed
        adata_RNA.layers['imputed'] = imputed_full



df_results = pd.DataFrame(all_results)
df_results.to_csv(os.path.join(RESULTS_DIR, "DualGate_DEC_benchmark_results.csv"), index=False)
print(f"\nAll per-seed results saved to {RESULTS_DIR}/DualGate_DEC_benchmark_results.csv")

# Comprehensive Summary Table: Mean ± Std for All Datasets
summary_records = []
timing_records = []

for d_name, group in df_results.groupby('dataset', sort=False):
    rec = {'Dataset': d_name, 'Seeds': len(group)}
    for m in ['ARI', 'NMI', 'AMI', 'Homogeneity', 'V-Measure', 'FMI', 'Silhouette', 'CHI', 'DBI']:
        if m in group.columns:
            rec[f"{m} (Mean ± Std)"] = f"{group[m].mean():.4f} ± {group[m].std():.4f}"
    if 'Execution_Time_s' in group.columns:
        dur_mean = group['Execution_Time_s'].mean()
        dur_std = group['Execution_Time_s'].std()
        dur_total = group['Execution_Time_s'].sum()
        rec['Training Time (s)'] = f"{dur_mean:.2f} ± {dur_std:.2f}"
        
        m_mins, m_secs = divmod(int(dur_mean), 60)
        t_mins, t_secs = divmod(int(dur_total), 60)
        timing_records.append({
            'Method': 'DualGate-DEC',
            'Dataset': d_name,
            'Num_Seeds': len(group),
            'Time_per_Seed_s': f"{dur_mean:.2f} ± {dur_std:.2f}",
            'Total_Dataset_Time_s': f"{dur_total:.2f}",
            'Formatted_Time_per_Seed': f"{m_mins}m {m_secs:02d}s" if m_mins > 0 else f"{dur_mean:.1f}s",
            'Formatted_Total_Time': f"{t_mins}m {t_secs:02d}s"
        })
    summary_records.append(rec)

summary_df = pd.DataFrame(summary_records)
summary_csv_path = os.path.join(RESULTS_DIR, "DualGate_DEC_benchmark_summary_mean_std.csv")
summary_df.to_csv(summary_csv_path, index=False)
print(f"Summary table saved to {summary_csv_path}\n")
print("=== FINAL UNIVERSAL BENCHMARK SUMMARY: DualGate-DEC (Mean ± Std across all Datasets) ===")
print(summary_df.to_string(index=False))

# Dedicated SOTA Runtime Comparison CSV
if timing_records:
    df_timing = pd.DataFrame(timing_records)
    timing_csv_path = os.path.join(RESULTS_DIR, "DualGate_DEC_training_time_summary.csv")
    df_timing.to_csv(timing_csv_path, index=False)
    print(f"\n[✓] Dedicated runtime comparison table saved to: {timing_csv_path}")

# =========================================================================
# FINAL EXECUTION TIME REPORT (FOR BENCHMARK & EFFICIENCY COMPARISON)
# =========================================================================
if 'Execution_Time_s' in df_results.columns:
    print("\n" + "=" * 88)
    print(" ⏱️ MODEL TRAINING EXECUTION TIME REPORT (FOR BENCHMARK COMPARISON) ".center(88, "="))
    print("=" * 88)
    total_all_seconds = float(df_results['Execution_Time_s'].sum())
    for d_name, group in df_results.groupby('dataset', sort=False):
        mean_dur = float(group['Execution_Time_s'].mean())
        std_dur = float(group['Execution_Time_s'].std())
        ds_total = float(group['Execution_Time_s'].sum())
        m_mins, m_secs = divmod(int(mean_dur), 60)
        d_mins, d_secs = divmod(int(ds_total), 60)
        print(f" • {d_name:<28} : {mean_dur:6.2f}s ± {std_dur:5.2f}s per seed (~{m_mins}m {m_secs:02d}s) | Total: {d_mins}m {d_secs:02d}s ({len(group)} seeds)")

    print("-" * 88)
    mean_all_dur = float(df_results['Execution_Time_s'].mean())
    std_all_dur = float(df_results['Execution_Time_s'].std())
    m_all_mins, m_all_secs = divmod(int(mean_all_dur), 60)
    all_mins, all_secs = divmod(int(total_all_seconds), 60)
    print(f" • OVERALL AVERAGE PER SEED RUN : {mean_all_dur:6.2f}s ± {std_all_dur:5.2f}s (~{m_all_mins}m {m_all_secs:02d}s per run)")
    print(f" • TOTAL WALL-CLOCK TRAINING TIME: {all_mins}m {all_secs:02d}s ({total_all_seconds:.2f} seconds total across all runs)")
    print("=" * 88 + "\n")


In [ ]:
# Cell 10: Cross-Seed Linearity & Metric Spread Suite (Side-by-Side: ARI vs Silhouette across Seeds)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Retrieve benchmark results from memory or saved CSV
RESULTS_DIR = os.path.join(OUTPUT_DIR, "results") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/results"
VISUALS_DIR = os.path.join(OUTPUT_DIR, "visuals") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/visuals"
os.makedirs(VISUALS_DIR, exist_ok=True)

csv_candidates = [
    os.path.join(RESULTS_DIR, "DualGate_DEC_benchmark_results.csv"),
    os.path.join(OUTPUT_DIR, "DualGate_DEC_benchmark_results.csv") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/DualGate_DEC_benchmark_results.csv"
]
csv_fallback = next((p for p in csv_candidates if os.path.exists(p)), None)
if 'df_results' not in locals() or df_results is None or len(df_results) == 0:
    if csv_fallback is not None and os.path.exists(csv_fallback):
        df_results = pd.read_csv(csv_fallback)
        print(f"[Loaded] Retrieved {len(df_results)} benchmark rows from: {csv_fallback}")
    else:
        print("[Notice] No benchmark results found. Please execute Cell 9 first.")
        df_results = None

if df_results is not None and len(df_results) > 0:
    unique_datasets = df_results['dataset'].unique()
    n_datasets = len(unique_datasets)

    fig, axes = plt.subplots(n_datasets, 2, figsize=(18, 4.8 * n_datasets), squeeze=False)

    print("\n" + "="*85)
    print(" CROSS-SEED LINEARITY & SPREAD DIAGNOSTICS (ARI & SILHOUETTE)")
    print("="*85)

    for idx, d_name in enumerate(unique_datasets):
        sub_df = df_results[df_results['dataset'] == d_name].copy()

        # Format seeds on X-axis as categorical labels to ensure uniform spacing
        sub_df['seed_str'] = sub_df['seed'].astype(str)
        seeds_order = sub_df['seed_str'].tolist()
        x_indices = np.arange(len(seeds_order))

        # -------------------------------------------------------------
        # Left Subplot: Adjusted Rand Index (ARI) across all seeds
        # -------------------------------------------------------------
        ari_vals = sub_df['ARI'].values
        mean_ari = float(np.mean(ari_vals))
        std_ari = float(np.std(ari_vals))
        min_ari = float(np.min(ari_vals))
        max_ari = float(np.max(ari_vals))
        range_ari = max_ari - min_ari

        ax_ari = axes[idx, 0]
        # Line + Marker plot across seeds
        ax_ari.plot(x_indices, ari_vals, marker='o', markersize=9, linewidth=2.4, color='#1f77b4', label='Seed ARI')
        # Horizontal reference line for Mean ARI (shows linearity/flatness)
        ax_ari.axhline(mean_ari, color='#d62728', linestyle='--', linewidth=2.0, label=f'Mean ARI: {mean_ari:.4f}')
        # Shaded ±1 Std confidence band
        ax_ari.fill_between(x_indices, mean_ari - std_ari, mean_ari + std_ari, color='#1f77b4', alpha=0.15, label=f'±1σ Band (±{std_ari:.4f})')
        # Annotate each seed point with its exact value
        for x_i, y_i in zip(x_indices, ari_vals):
            ax_ari.annotate(f"{y_i:.3f}", (x_i, y_i), textcoords="offset points", xytext=(0, 9), ha='center', fontsize=9, fontweight='bold', color='#1f77b4')

        ax_ari.set_xticks(x_indices)
        ax_ari.set_xticklabels([f"Seed {s}" for s in seeds_order], fontsize=10, rotation=25)
        ax_ari.set_ylabel("Adjusted Rand Index (ARI)", fontsize=11, fontweight='bold')
        ax_ari.set_title(f"{d_name}\n[ARI Linearity] Mean: {mean_ari:.4f} ± {std_ari:.4f} | Spread (Max-Min): {range_ari:.4f}", fontsize=12, fontweight='bold')
        ax_ari.grid(True, linestyle=':', alpha=0.6)
        ax_ari.legend(loc='lower right', fontsize=9)
        y_pad = max(0.04, range_ari * 0.4)
        ax_ari.set_ylim([max(0.0, min_ari - y_pad), min(1.0, max_ari + y_pad)])

        # -------------------------------------------------------------
        # Right Subplot: Silhouette Score across all seeds
        # -------------------------------------------------------------
        sil_vals = sub_df['Silhouette'].values
        mean_sil = float(np.mean(sil_vals))
        std_sil = float(np.std(sil_vals))
        min_sil = float(np.min(sil_vals))
        max_sil = float(np.max(sil_vals))
        range_sil = max_sil - min_sil

        ax_sil = axes[idx, 1]
        # Line + Marker plot across seeds
        ax_sil.plot(x_indices, sil_vals, marker='s', markersize=9, linewidth=2.4, color='#2ca02c', label='Seed Silhouette')
        # Horizontal reference line for Mean Silhouette
        ax_sil.axhline(mean_sil, color='#9467bd', linestyle='--', linewidth=2.0, label=f'Mean Sil: {mean_sil:.4f}')
        # Shaded ±1 Std confidence band
        ax_sil.fill_between(x_indices, mean_sil - std_sil, mean_sil + std_sil, color='#2ca02c', alpha=0.15, label=f'±1σ Band (±{std_sil:.4f})')
        # Annotate each seed point with its exact value
        for x_i, y_i in zip(x_indices, sil_vals):
            ax_sil.annotate(f"{y_i:.3f}", (x_i, y_i), textcoords="offset points", xytext=(0, 9), ha='center', fontsize=9, fontweight='bold', color='#2ca02c')

        ax_sil.set_xticks(x_indices)
        ax_sil.set_xticklabels([f"Seed {s}" for s in seeds_order], fontsize=10, rotation=25)
        ax_sil.set_ylabel("Silhouette Score", fontsize=11, fontweight='bold')
        ax_sil.set_title(f"{d_name}\n[Silhouette Linearity] Mean: {mean_sil:.4f} ± {std_sil:.4f} | Spread (Max-Min): {range_sil:.4f}", fontsize=12, fontweight='bold')
        ax_sil.grid(True, linestyle=':', alpha=0.6)
        ax_sil.legend(loc='lower right', fontsize=9)
        y_pad_sil = max(0.04, range_sil * 0.4)
        ax_sil.set_ylim([max(-1.0, min_sil - y_pad_sil), min(1.0, max_sil + y_pad_sil)])

        # Detailed Linearity Diagnostic
        print(f"Dataset: {d_name:<30} | Seeds Tested: {len(seeds_order)}")
        print(f"   • ARI Spread        -> Mean: {mean_ari:.4f} | Std: {std_ari:.4f} | Min: {min_ari:.4f} | Max: {max_ari:.4f} | Spread (Max-Min): {range_ari:.4f}")
        print(f"   • Silhouette Spread -> Mean: {mean_sil:.4f} | Std: {std_sil:.4f} | Min: {min_sil:.4f} | Max: {max_sil:.4f} | Spread (Max-Min): {range_sil:.4f}")
        if range_ari <= 0.03:
            print("   ► Linearity Verdict : Ultra-stable flat line (Near-zero variance, high seed independence).")
        elif range_ari <= 0.06:
            print("   ► Linearity Verdict : Near-linear trajectory across seeds (Good stability).")
        else:
            print(f"   ► Linearity Verdict : Noticeable variation across seeds ({range_ari*100:.1f}% spread).")

    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.subplots_adjust(top=0.96, hspace=0.35)
    plt.suptitle("DualGate-DEC: Side-by-Side Cross-Seed Linearity & Spread Diagnostics (ARI & Silhouette)", fontsize=16, fontweight='bold', y=0.995)
    save_path = os.path.join(VISUALS_DIR, "DualGate_DEC_cross_seed_linearity.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n[Saved] Unified cross-seed linearity figure saved to: {save_path}")
    plt.show()


In [ ]:
# Cell 11: Per-Epoch Visual Analytics Suite (Unified Multi-Dataset Trajectory)
import os
import matplotlib.pyplot as plt
import numpy as np

VISUALS_DIR = os.path.join(OUTPUT_DIR, "visuals") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/visuals"
os.makedirs(VISUALS_DIR, exist_ok=True)

if 'all_histories' not in locals() or all_histories is None or len(all_histories) == 0:
    print("[Notice] No per-epoch training histories found in memory. Please execute Cell 9 first.")
else:
    dataset_names = list(all_histories.keys())
    n_datasets = len(dataset_names)
    epochs_total = PRETRAIN_EPOCHS + FINETUNE_EPOCHS
    x_epochs = np.arange(1, epochs_total + 1)

    print("\n" + "="*85)
    print(f" PER-EPOCH MULTI-LOSS & SILHOUETTE CONVERGENCE TRAJECTORIES ({n_datasets} DATASETS)")
    print("="*85)

    # Render ONE unified figure containing all datasets (n_datasets rows x 3 columns)
    fig, axes = plt.subplots(n_datasets, 3, figsize=(21, 5.0 * n_datasets), squeeze=False)

    for idx, d_name in enumerate(dataset_names):
        seeds_dict = all_histories[d_name]

        # -------------------------------------------------------------
        # 1. Total & Reconstruction Losses (Col 0)
        # -------------------------------------------------------------
        for s, hist in seeds_dict.items():
            axes[idx, 0].plot(x_epochs, hist['total_loss'], alpha=0.25, color='#1f77b4', lw=1.2)
        mean_tot = np.mean([hist['total_loss'] for hist in seeds_dict.values()], axis=0)
        axes[idx, 0].plot(x_epochs, mean_tot, color='#1f77b4', lw=2.4, label=f'Mean Total Loss (Final: {mean_tot[-1]:.4f})')
        axes[idx, 0].axvline(x=PRETRAIN_EPOCHS, color='#d62728', linestyle='--', lw=1.8, label=f'DEC Phase Activated (Ep {PRETRAIN_EPOCHS})')
        axes[idx, 0].set_title(f"{d_name}\nTotal Convergence Loss (Stage 1 + Stage 2)", fontsize=12, fontweight='bold')
        axes[idx, 0].set_xlabel("Epoch (180 Total)", fontsize=10)
        axes[idx, 0].set_ylabel("Objective Loss", fontsize=10)
        axes[idx, 0].legend(loc='upper right', fontsize=8.5)
        axes[idx, 0].grid(True, linestyle=':', alpha=0.5)

        # -------------------------------------------------------------
        # 2. Stage 2 DEC KL Divergence Loss (Col 1)
        # -------------------------------------------------------------
        stage2_epochs = x_epochs[PRETRAIN_EPOCHS:]
        for s, hist in seeds_dict.items():
            axes[idx, 1].plot(stage2_epochs, hist['kl_loss'][PRETRAIN_EPOCHS:], alpha=0.3, color='#e377c2', lw=1.2)
        mean_kl = np.mean([hist['kl_loss'][PRETRAIN_EPOCHS:] for hist in seeds_dict.values()], axis=0)
        axes[idx, 1].plot(stage2_epochs, mean_kl, color='#d62728', lw=2.4, label=f'Mean Potts-DEC KL (Final: {mean_kl[-1]:.4f})')
        axes[idx, 1].set_title(f"{d_name}\nSpatial Potts DEC KL Divergence Loss", fontsize=12, fontweight='bold')
        axes[idx, 1].set_xlabel(f"Epoch (Stage 2: {PRETRAIN_EPOCHS+1}-{epochs_total})", fontsize=10)
        axes[idx, 1].set_ylabel("KL Divergence", fontsize=10)
        axes[idx, 1].legend(loc='upper right', fontsize=8.5)
        axes[idx, 1].grid(True, linestyle=':', alpha=0.5)

        # -------------------------------------------------------------
        # 3. Continuous Silhouette Trajectory across ALL 180 Epochs (Col 2)
        # -------------------------------------------------------------
        for s, hist in seeds_dict.items():
            axes[idx, 2].plot(x_epochs, hist['silhouette'], alpha=0.25, color='#2ca02c', lw=1.2)
        mean_sil = np.mean([hist['silhouette'] for hist in seeds_dict.values()], axis=0)
        axes[idx, 2].plot(x_epochs, mean_sil, color='#2ca02c', lw=2.4, label='Mean Silhouette Trajectory')
        axes[idx, 2].axvline(x=PRETRAIN_EPOCHS, color='#d62728', linestyle='--', lw=1.8, label=f'DEC Phase Activated (Ep {PRETRAIN_EPOCHS})')

        best_mean_ep = int(np.argmax(mean_sil)) + 1
        best_mean_sil = float(mean_sil[best_mean_ep - 1])
        axes[idx, 2].scatter([best_mean_ep], [best_mean_sil], color='red', s=80, zorder=5, label=f'Peak Sil: {best_mean_sil:.4f} @ Ep {best_mean_ep}')
        axes[idx, 2].set_title(f"{d_name}\nContinuous Latent Silhouette Trajectory", fontsize=12, fontweight='bold')
        axes[idx, 2].set_xlabel("Epoch (180 Total)", fontsize=10)
        axes[idx, 2].set_ylabel("Silhouette Score", fontsize=10)
        axes[idx, 2].legend(loc='lower right', fontsize=8.5)
        axes[idx, 2].grid(True, linestyle=':', alpha=0.5)

        print(f"Dataset: {d_name:<30} | Seeds: {len(seeds_dict)} | Peak Silhouette: {best_mean_sil:.4f} (Epoch {best_mean_ep}) | Final KL: {mean_kl[-1]:.4f}")

    # Unified Overarching Title
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.subplots_adjust(top=0.96, hspace=0.35)
    plt.suptitle("DualGate-DEC: Per-Epoch Loss Convergence & Continuous Silhouette Trajectories across Datasets (180 Epochs)", fontsize=16, fontweight='bold', y=0.995)
    save_path = os.path.join(VISUALS_DIR, "DualGate_DEC_epoch_trajectories.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n[Saved] Unified per-epoch analytics figure saved to: {save_path}")
    plt.show()


In [ ]:
# Cell 12: Side-by-Side Spatial Domain & Attention Map Visualization for ALL Datasets (Unified Figure)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc

RESULTS_DIR = os.path.join(OUTPUT_DIR, "results") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/results"
VISUALS_DIR = os.path.join(OUTPUT_DIR, "visuals") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/visuals"
os.makedirs(VISUALS_DIR, exist_ok=True)

datasets_to_plot = []
if 'all_dataset_adata' in locals() and len(all_dataset_adata) > 0:
    datasets_to_plot = list(all_dataset_adata.keys())
elif 'all_results' in locals() and len(all_results) > 0:
    datasets_to_plot = list(pd.DataFrame(all_results)['dataset'].unique())
elif 'all_histories' in locals() and len(all_histories) > 0:
    datasets_to_plot = list(all_histories.keys())
else:
    datasets_to_plot = [CHOICES[i][0] for i in DATASET_INDICES]

# Filter datasets that have loadable adata
loaded_adatas = {}
for d_name in datasets_to_plot:
    adata_plot = None
    if 'all_dataset_adata' in locals() and d_name in all_dataset_adata:
        adata_plot = all_dataset_adata[d_name]
    else:
        candidate_paths = [
            os.path.join(RESULTS_DIR, f"DualGate_DEC_adata_{d_name}.h5ad"),
            os.path.join(OUTPUT_DIR, f"DualGate_DEC_adata_{d_name}.h5ad") if 'OUTPUT_DIR' in locals() else "",
            f"DualGate_DEC_outputs/results/DualGate_DEC_adata_{d_name}.h5ad",
            f"DualGate_DEC_outputs/DualGate_DEC_adata_{d_name}.h5ad"
        ]
        for p in candidate_paths:
            if p and os.path.exists(p):
                try:
                    adata_plot = sc.read_h5ad(p)
                    break
                except Exception:
                    pass

    if adata_plot is None and 'dataset_name' in locals() and dataset_name == d_name:
        adata_plot = adata_hvg if 'adata_hvg' in locals() else adata_RNA

    if adata_plot is not None:
        loaded_adatas[d_name] = adata_plot

if len(loaded_adatas) == 0:
    print("[Notice] No spatial domain AnnData objects available to render. Please run Cell 9 first.")
else:
    valid_datasets = list(loaded_adatas.keys())
    n_datasets = len(valid_datasets)

    print("\n" + "="*85)
    print(f" SPATIAL DOMAIN PARTITIONING & MODALITY ATTENTION VISUALIZATION ({n_datasets} DATASETS)")
    print("="*85)

    # Render ONE unified figure containing all datasets (n_datasets rows x 3 columns)
    # Uses natural auto-aspect (without set_aspect('equal')) to prevent A1/D1 from shrinking
    fig, axes = plt.subplots(n_datasets, 3, figsize=(21, 5.2 * n_datasets), squeeze=False)

    for row_idx, d_name in enumerate(valid_datasets):
        adata_plot = loaded_adatas[d_name]

        coords = adata_plot.obsm.get('spatial', None)
        if coords is None:
            if 'dataset_name' in locals() and dataset_name == d_name and 'cell_positions' in locals():
                coords = cell_positions
            else:
                coords = np.zeros((adata_plot.n_obs, 2))

        gt_series = adata_plot.obs.get('ground_truth', None)
        gt_labels = pd.factorize(gt_series)[0] if gt_series is not None else np.zeros(coords.shape[0])

        domain_col = 'DualGate_DEC_domain' if 'DualGate_DEC_domain' in adata_plot.obs else ('domain' if 'domain' in adata_plot.obs else None)
        pred_labels = pd.factorize(adata_plot.obs[domain_col])[0] if (domain_col and domain_col in adata_plot.obs) else gt_labels

        has_att = 'mean_rna_attention' in adata_plot.obs
        mean_att = adata_plot.obs['mean_rna_attention'].values if has_att else None

        # Retrieve exact seed and benchmark metrics for panel subtitle
        row_metric_str = ""
        seed_label = adata_plot.uns.get('selected_seed', None)
        if 'df_results' in locals() and df_results is not None:
            sub_m = df_results[df_results['dataset'] == d_name]
            if len(sub_m) > 0:
                if seed_label is not None and (sub_m['seed'] == seed_label).any():
                    row_s = sub_m[sub_m['seed'] == seed_label].iloc[0]
                    row_metric_str = f" (Best ARI Seed: {seed_label} | ARI: {row_s['ARI']:.4f}, Sil: {row_s['Silhouette']:.4f})"
                else:
                    top_row = sub_m.sort_values(by='ARI', ascending=False).iloc[0]
                    row_metric_str = f" (Best ARI Seed: {top_row['seed']} | ARI: {top_row['ARI']:.4f}, Sil: {top_row['Silhouette']:.4f})"
        elif seed_label is not None:
            row_metric_str = f" (Seed: {seed_label})"

        # -------------------------------------------------------------
        # 1. Ground Truth Annotations (Col 0)
        # -------------------------------------------------------------
        ax0 = axes[row_idx, 0]
        is_bg = (gt_labels == -1)
        if is_bg.any():
            ax0.scatter(coords[is_bg, 0], coords[is_bg, 1], c='#e0e0e0', s=20, alpha=0.35, label='Unannotated')
        valid_mask = ~is_bg
        ax0.scatter(coords[valid_mask, 0], coords[valid_mask, 1], c=gt_labels[valid_mask], cmap='tab20', s=20, alpha=0.9)
        ax0.set_title(f"{d_name}\nGround Truth Annotations", fontsize=12, fontweight='bold')
        ax0.invert_yaxis()
        ax0.axis('off')

        # -------------------------------------------------------------
        # 2. DualGate-DEC Spatial Domains (Col 1)
        # -------------------------------------------------------------
        ax1 = axes[row_idx, 1]
        ax1.scatter(coords[:, 0], coords[:, 1], c=pred_labels, cmap='tab20', s=20, alpha=0.9)
        ax1.set_title(f"{d_name}\nDualGate-DEC Spatial Domains{row_metric_str}", fontsize=12, fontweight='bold')
        ax1.invert_yaxis()
        ax1.axis('off')

        # -------------------------------------------------------------
        # 3. Spatial Modality Attention Heatmap (Col 2)
        # -------------------------------------------------------------
        ax2 = axes[row_idx, 2]
        if has_att and mean_att is not None:
            sc_att = ax2.scatter(coords[:, 0], coords[:, 1], c=mean_att, cmap='coolwarm', s=20, vmin=0.0, vmax=1.0, alpha=0.95)
            cbar = plt.colorbar(sc_att, ax=ax2, fraction=0.046, pad=0.04)
            cbar.set_label('RNA Attention Weight (1 - Aux Weight)', fontsize=9, fontweight='bold')
            ax2.set_title(f"{d_name}\nModality Attention Map (RNA vs Aux)", fontsize=12, fontweight='bold')
        else:
            ax2.text(0.5, 0.5, "Attention weights not extracted", ha='center', va='center', fontsize=12, color='gray')
            ax2.set_title(f"{d_name}\nModality Attention Map", fontsize=12, fontweight='bold')
        ax2.invert_yaxis()
        ax2.axis('off')

        print(f"Dataset: {d_name:<30} | Cells/Spots: {coords.shape[0]} | Domains Found: {len(np.unique(pred_labels))} | Attention Mapped: {has_att}")

    # Unified overarching title with ample top padding
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.subplots_adjust(top=0.96, hspace=0.35)
    plt.suptitle("DualGate-DEC: Side-by-Side Spatial Domain Partitioning & Modality Attention Heatmaps across Datasets", fontsize=16, fontweight='bold', y=0.995)

    # Save ONE unified figure containing all datasets to the visuals directory
    save_path = os.path.join(VISUALS_DIR, "DualGate_DEC_spatial_domains_and_attention.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n[Saved] Unified spatial domains & attention map figure ({n_datasets} datasets) saved to: {save_path}")
    plt.show()

print("\n" + "="*85)
print(" SPATIAL DOMAIN & ATTENTION VISUALIZATION COMPLETE")
print("="*85 + "\n")


In [ ]:
# Cell 13: Latent Embedding Manifold Projections (Side-by-Side UMAP Suite for ALL Datasets)
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
from mpl_toolkits.axes_grid1 import make_axes_locatable

RESULTS_DIR = os.path.join(OUTPUT_DIR, "results") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/results"
VISUALS_DIR = os.path.join(OUTPUT_DIR, "visuals") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/visuals"
os.makedirs(VISUALS_DIR, exist_ok=True)

# Resolve datasets to render
datasets_to_plot = []
if 'all_dataset_adata' in locals() and len(all_dataset_adata) > 0:
    datasets_to_plot = list(all_dataset_adata.keys())
elif 'all_results' in locals() and len(all_results) > 0:
    datasets_to_plot = list(pd.DataFrame(all_results)['dataset'].unique())
elif 'all_histories' in locals() and len(all_histories) > 0:
    datasets_to_plot = list(all_histories.keys())
else:
    datasets_to_plot = [CHOICES[i][0] for i in DATASET_INDICES]

# Filter datasets that have loadable AnnData with embeddings
loaded_adatas = {}
for d_name in datasets_to_plot:
    adata_plot = None
    if 'all_dataset_adata' in locals() and d_name in all_dataset_adata:
        adata_plot = all_dataset_adata[d_name]
    else:
        candidate_paths = [
            os.path.join(RESULTS_DIR, f"DualGate_DEC_adata_{d_name}.h5ad"),
            os.path.join(OUTPUT_DIR, f"DualGate_DEC_adata_{d_name}.h5ad") if 'OUTPUT_DIR' in locals() else "",
            f"DualGate_DEC_outputs/results/DualGate_DEC_adata_{d_name}.h5ad",
            f"DualGate_DEC_outputs/DualGate_DEC_adata_{d_name}.h5ad"
        ]
        for p in candidate_paths:
            if p and os.path.exists(p):
                try:
                    adata_plot = sc.read_h5ad(p)
                    break
                except Exception:
                    pass

    if adata_plot is not None and 'DualGate_DEC_emb' in adata_plot.obsm:
        loaded_adatas[d_name] = adata_plot

if len(loaded_adatas) == 0:
    print("[Notice] No latent embeddings found to compute UMAP. Please execute Cell 9 first.")
else:
    valid_datasets = list(loaded_adatas.keys())
    n_datasets = len(valid_datasets)

    print("\n" + "="*85)
    print(f" LATENT EMBEDDING MANIFOLD VISUALIZATION (UMAP) ({n_datasets} DATASETS - BEST ARI SEED)")
    print("="*85)

    # Compute UMAP for each dataset if not already precomputed
    for d_name in valid_datasets:
        adata_curr = loaded_adatas[d_name]
        if 'X_umap' not in adata_curr.obsm:
            print(f"   • Computing 2D UMAP projection for {d_name}...")
            sc.pp.neighbors(adata_curr, use_rep='DualGate_DEC_emb', n_neighbors=15, metric='cosine', random_state=42)
            sc.tl.umap(adata_curr, min_dist=0.3, spread=1.0, random_state=42)

    # Render ONE unified figure containing all datasets (n_datasets rows x 3 columns)
    fig, axes = plt.subplots(n_datasets, 3, figsize=(21, 5.8 * n_datasets), squeeze=False)

    for row_idx, d_name in enumerate(valid_datasets):
        adata_plot = loaded_adatas[d_name]
        umap_coords = adata_plot.obsm['X_umap']

        gt_series = adata_plot.obs.get('ground_truth', None)
        gt_labels = pd.factorize(gt_series)[0] if gt_series is not None else np.zeros(umap_coords.shape[0])

        domain_col = 'DualGate_DEC_domain' if 'DualGate_DEC_domain' in adata_plot.obs else ('domain' if 'domain' in adata_plot.obs else None)
        pred_labels = pd.factorize(adata_plot.obs[domain_col])[0] if (domain_col and domain_col in adata_plot.obs) else gt_labels

        has_att = 'mean_rna_attention' in adata_plot.obs
        mean_att = adata_plot.obs['mean_rna_attention'].values if has_att else None

        # Retrieve exact best-ARI seed and benchmark metrics for panel subtitle
        row_metric_str = ""
        seed_label = adata_plot.uns.get('selected_seed', None)
        if 'df_results' in locals() and df_results is not None:
            sub_m = df_results[df_results['dataset'] == d_name]
            if len(sub_m) > 0:
                if seed_label is not None and (sub_m['seed'] == seed_label).any():
                    row_s = sub_m[sub_m['seed'] == seed_label].iloc[0]
                    row_metric_str = f" (Best ARI Seed: {seed_label} | ARI: {row_s['ARI']:.4f}, Sil: {row_s['Silhouette']:.4f})"
                else:
                    top_row = sub_m.sort_values(by='ARI', ascending=False).iloc[0]
                    row_metric_str = f" (Best ARI Seed: {top_row['seed']} | ARI: {top_row['ARI']:.4f} | Sil: {top_row['Silhouette']:.4f})"
        elif seed_label is not None:
            row_metric_str = f" (Best ARI Seed: {seed_label})"

        # -------------------------------------------------------------
        # 1. Ground Truth Annotations in Latent UMAP (Col 0)
        # -------------------------------------------------------------
        ax0 = axes[row_idx, 0]
        is_bg = (gt_labels == -1)
        if is_bg.any():
            ax0.scatter(umap_coords[is_bg, 0], umap_coords[is_bg, 1], c='#e0e0e0', s=16, alpha=0.35, label='Unannotated')
        valid_mask = ~is_bg
        ax0.scatter(umap_coords[valid_mask, 0], umap_coords[valid_mask, 1], c=gt_labels[valid_mask], cmap='tab20', s=22, alpha=0.85)
        ax0.set_title(f"{d_name}\nLatent UMAP: Ground Truth Annotations", fontsize=12, fontweight='bold')
        ax0.set_xlabel("UMAP 1", fontsize=10)
        ax0.set_ylabel("UMAP 2", fontsize=10)
        ax0.grid(True, linestyle=':', alpha=0.4)

        # -------------------------------------------------------------
        # 2. DualGate-DEC Discovered Spatial Domains in Latent UMAP (Col 1)
        # -------------------------------------------------------------
        ax1 = axes[row_idx, 1]
        ax1.scatter(umap_coords[:, 0], umap_coords[:, 1], c=pred_labels, cmap='tab20', s=22, alpha=0.85)
        ax1.set_title(f"{d_name}\nLatent UMAP: DualGate-DEC Domains{row_metric_str}", fontsize=12, fontweight='bold')
        ax1.set_xlabel("UMAP 1", fontsize=10)
        ax1.set_ylabel("UMAP 2", fontsize=10)
        ax1.grid(True, linestyle=':', alpha=0.4)

        # -------------------------------------------------------------
        # 3. Spatial Modality Attention Heatmap on Latent UMAP (Col 2)
        # -------------------------------------------------------------
        ax2 = axes[row_idx, 2]
        if has_att and mean_att is not None:
            sc_att = ax2.scatter(umap_coords[:, 0], umap_coords[:, 1], c=mean_att, cmap='coolwarm', s=22, vmin=0.0, vmax=1.0, alpha=0.9)
            divider = make_axes_locatable(ax2)
            cax = divider.append_axes("right", size="5%", pad=0.08)
            cbar = fig.colorbar(sc_att, cax=cax)
            cbar.set_label('RNA Attention Weight α_rna', fontsize=9, fontweight='bold')
            ax2.set_title(f"{d_name}\nLatent UMAP: Modality Attention (RNA vs Aux)", fontsize=12, fontweight='bold')
        else:
            ax2.text(0.5, 0.5, "Attention weights not extracted", ha='center', va='center', fontsize=12, color='gray')
            ax2.set_title(f"{d_name}\nLatent UMAP: Modality Attention", fontsize=12, fontweight='bold')
        ax2.set_xlabel("UMAP 1", fontsize=10)
        ax2.set_ylabel("UMAP 2", fontsize=10)
        ax2.grid(True, linestyle=':', alpha=0.4)

        print(f"Dataset: {d_name:<30} | Manifold Spots: {umap_coords.shape[0]} | Clusters: {len(np.unique(pred_labels))} | Attention Mapped: {has_att}")

    # Unified Overarching Title with breathing room
    plt.tight_layout(rect=[0, 0, 1, 0.985])
    plt.subplots_adjust(top=0.96, hspace=0.35)
    plt.suptitle("DualGate-DEC: Latent Embedding Manifold Projections (UMAP) across Datasets (Best ARI Seed)", fontsize=16, fontweight='bold', y=0.995)
    save_path = os.path.join(VISUALS_DIR, "DualGate_DEC_latent_umap.png")
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    print(f"\n[Saved] Unified latent UMAP manifold figure saved to: {save_path}")
    plt.show()


In [ ]:
# Cell 14: Dedicated Biological Biomarker Discovery & Statistical Differential Expression
import os
import scanpy as sc
import pandas as pd
import numpy as np
import glob
from IPython.display import display

RESULTS_DIR = os.path.join(OUTPUT_DIR, "results") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/results"
BIOMARKERS_DIR = os.path.join(OUTPUT_DIR, "biomarkers") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/biomarkers"
os.makedirs(BIOMARKERS_DIR, exist_ok=True)

print("="*85)
print(" BIOLOGICAL BIOMARKER DISCOVERY & STATISTICAL DIFFERENTIAL EXPRESSION")
print("="*85)

# 1. Resolve datasets to analyze (from memory or saved files on disk)
datasets_to_analyze = []
if 'all_dataset_adata' in locals() and len(all_dataset_adata) > 0:
    datasets_to_analyze = list(all_dataset_adata.keys())
elif os.path.exists(RESULTS_DIR) or ('OUTPUT_DIR' in locals() and os.path.exists(OUTPUT_DIR)):
    h5ad_files = glob.glob(os.path.join(RESULTS_DIR, "DualGate_DEC_adata_*.h5ad")) or glob.glob(os.path.join(OUTPUT_DIR, "DualGate_DEC_adata_*.h5ad"))
    for f in h5ad_files:
        d_name = os.path.basename(f).replace("DualGate_DEC_adata_", "").replace(".h5ad", "")
        datasets_to_analyze.append(d_name)

if len(datasets_to_analyze) == 0 and 'CHOICES' in locals() and 'DATASET_INDICES' in locals():
    datasets_to_analyze = [CHOICES[i][0] for i in DATASET_INDICES]

print(f"[Biomarker Engine] Analyzing {len(datasets_to_analyze)} dataset(s): {datasets_to_analyze}\n")

all_markers_dict = {}

# 2. Compute Wilcoxon differential expression for each dataset
for d_name in datasets_to_analyze:
    print(f"\n{'#'*85}\n DATASET: {d_name} — Discovered Spatial Domain Biomarkers\n{'#'*85}")

    adata_curr = None
    if 'all_dataset_adata' in locals() and d_name in all_dataset_adata:
        adata_curr = all_dataset_adata[d_name].copy()
    else:
        candidate_h5ad = [
            os.path.join(RESULTS_DIR, f"DualGate_DEC_adata_{d_name}.h5ad"),
            os.path.join(OUTPUT_DIR, f"DualGate_DEC_adata_{d_name}.h5ad") if 'OUTPUT_DIR' in locals() else ""
        ]
        h5ad_path = next((p for p in candidate_h5ad if p and os.path.exists(p)), None)
        if h5ad_path is not None and os.path.exists(h5ad_path):
            adata_curr = sc.read_h5ad(h5ad_path)

    if adata_curr is None:
        print(f"[Notice] Data for {d_name} not found. Skipping.")
        continue

    domain_col = 'DualGate_DEC_domain' if 'DualGate_DEC_domain' in adata_curr.obs else ('domain' if 'domain' in adata_curr.obs else None)
    if domain_col is None or domain_col not in adata_curr.obs:
        print(f"[Warning] Domain column not found in adata.obs for {d_name}. Skipping.")
        continue

    # Ensure variable names are strictly unique to avoid scanpy dotplot errors
    adata_curr.var_names_make_unique()

    # Ensure domain labels are categorical
    adata_curr.obs[domain_col] = pd.Categorical(adata_curr.obs[domain_col].astype(str))
    
    # Filter out tiny singleton clusters (< 3 spots) to avoid Wilcoxon variance errors (cures D1)
    domain_counts = adata_curr.obs[domain_col].value_counts()
    min_spots = 3
    valid_domains = [str(g) for g, count in domain_counts.items() if count >= min_spots]
    tiny_domains = [str(g) for g, count in domain_counts.items() if count < min_spots]
    
    if len(tiny_domains) > 0:
        print(f"[Filter] Excluded {len(tiny_domains)} tiny cluster(s) with < {min_spots} spots from Wilcoxon testing: {tiny_domains}")

    # Select layer (use denoised imputed layer if available, else normalized .X)
    use_layer = 'imputed' if 'imputed' in adata_curr.layers else None
    layer_name = 'Denoised Imputed Expression' if use_layer == 'imputed' else 'Normalized Expression (.X)'
    print(f"[Analysis] Computing Wilcoxon Rank-Sum Tests across {len(valid_domains)} domains using [{layer_name}]...")

    try:
        sc.tl.rank_genes_groups(
            adata_curr,
            groupby=domain_col,
            groups=valid_domains,
            layer=use_layer,
            method='wilcoxon',
            pts=True
        )

        marker_df = sc.get.rank_genes_groups_df(adata_curr, group=None)

        # Save full biomarker table to CSV
        save_path = os.path.join(BIOMARKERS_DIR, f"DualGate_DEC_markers_{d_name}.csv")
        marker_df.to_csv(save_path, index=False)
        print(f"[Saved] Full marker table ({len(marker_df):,} rows) exported to: {save_path}\n")

        all_markers_dict[d_name] = marker_df

        # -------------------------------------------------------------
        # Clean Formatted DataFrame Display (Colab / Jupyter Interactive Table)
        # -------------------------------------------------------------
        unique_groups = sorted(marker_df['group'].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))
        summary_rows = []

        for grp in unique_groups:
            sub = marker_df[marker_df['group'] == grp].head(5)
            n_spots = int((adata_curr.obs[domain_col] == grp).sum())
            pct_spots = (n_spots / adata_curr.n_obs) * 100.0

            row_dict = {
                'Spatial Domain': f"Domain {grp}",
                'Spots (Count / %)': f"{n_spots:,} ({pct_spots:.1f}%)"
            }
            for rank, (_, row) in enumerate(sub.iterrows(), 1):
                gene = row['names']
                score = row['scores']
                logfc = row.get('logfoldchanges', None)
                if logfc is not None and not np.isnan(logfc):
                    row_dict[f"Top {rank} Marker"] = f"{gene} (Z={score:.1f}, FC={logfc:.2f})"
                else:
                    row_dict[f"Top {rank} Marker"] = f"{gene} (Z={score:.1f})"

            summary_rows.append(row_dict)

        df_summary = pd.DataFrame(summary_rows)
        print(f"--- Top 5 Distinguishing Biomarkers Table: {d_name} ---")
        try:
            display(df_summary)
        except Exception:
            print(df_summary.to_string(index=False))

    except Exception as e:
        print(f"[Error] Biomarker Wilcoxon test failed for {d_name}: {e}")

print("\n" + "="*85)
print(" BIOMARKER DISCOVERY & CSV EXPORTS COMPLETE FOR ALL DATASETS")
print("="*85 + "\n")


In [ ]:
# Cell 15: Biomarker Visual Analytics (Marker Gene DotPlots & Layer Concordance Heatmaps)
import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import glob
from mpl_toolkits.axes_grid1 import make_axes_locatable

RESULTS_DIR = os.path.join(OUTPUT_DIR, "results") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/results"
BIOMARKERS_DIR = os.path.join(OUTPUT_DIR, "biomarkers") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/biomarkers"
VISUALS_DIR = os.path.join(OUTPUT_DIR, "visuals") if 'OUTPUT_DIR' in locals() else "DualGate_DEC_outputs/visuals"
os.makedirs(VISUALS_DIR, exist_ok=True)

print("="*85)
print(" BIOMARKER VISUAL ANALYTICS: STANDARDIZED DOTPLOTS & CONCORDANCE HEATMAPS")
print("="*85)

datasets_to_visualize = []
if 'all_markers_dict' in locals() and len(all_markers_dict) > 0:
    datasets_to_visualize = list(all_markers_dict.keys())
elif 'all_dataset_adata' in locals() and len(all_dataset_adata) > 0:
    datasets_to_visualize = list(all_dataset_adata.keys())
else:
    csv_files = glob.glob(os.path.join(BIOMARKERS_DIR, "DualGate_DEC_markers_*.csv"))
    for f in csv_files:
        d_name = os.path.basename(f).replace("DualGate_DEC_markers_", "").replace(".csv", "")
        datasets_to_visualize.append(d_name)

print(f"[Visualization Engine] Rendering biomarker figures for {len(datasets_to_visualize)} dataset(s): {datasets_to_visualize}\n")

for d_name in datasets_to_visualize:
    print(f"\n{'#'*85}\n RENDERING BIOMARKER FIGURES: {d_name}\n{'#'*85}")

    # Retrieve marker table
    marker_df = None
    if 'all_markers_dict' in locals() and d_name in all_markers_dict:
        marker_df = all_markers_dict[d_name]
    else:
        csv_p = os.path.join(BIOMARKERS_DIR, f"DualGate_DEC_markers_{d_name}.csv")
        if os.path.exists(csv_p):
            marker_df = pd.read_csv(csv_p)

    # Retrieve AnnData
    adata_curr = None
    if 'all_dataset_adata' in locals() and d_name in all_dataset_adata:
        adata_curr = all_dataset_adata[d_name].copy()
    else:
        candidate_h5ad = [
            os.path.join(RESULTS_DIR, f"DualGate_DEC_adata_{d_name}.h5ad"),
            os.path.join(OUTPUT_DIR, f"DualGate_DEC_adata_{d_name}.h5ad") if 'OUTPUT_DIR' in locals() else ""
        ]
        h5ad_path = next((p for p in candidate_h5ad if p and os.path.exists(p)), None)
        if h5ad_path is not None and os.path.exists(h5ad_path):
            adata_curr = sc.read_h5ad(h5ad_path)

    if adata_curr is None or marker_df is None:
        print(f"[Notice] Data or markers for {d_name} not available. Skipping.")
        continue

    adata_curr.var_names_make_unique()
    domain_col = 'DualGate_DEC_domain' if 'DualGate_DEC_domain' in adata_curr.obs else ('domain' if 'domain' in adata_curr.obs else None)

    # Extract top 3 distinguishing marker genes per domain for the DotPlot (numerically sorted)
    unique_groups = sorted(marker_df['group'].unique(), key=lambda x: int(x) if str(x).isdigit() else str(x))
    top_markers_dict = {}
    for grp in unique_groups:
        sub = marker_df[marker_df['group'] == grp]
        valid_genes = [g for g in sub['names'] if g in adata_curr.var_names][:3]
        if len(valid_genes) > 0:
            top_markers_dict[f"Domain {grp}"] = valid_genes

    # -------------------------------------------------------------
    # 1. Standardized Publication DotPlot (Uniform figsize=(14, 5.5))
    # -------------------------------------------------------------
    try:
        if len(top_markers_dict) > 0:
            print(f"[1/2] Generating Standardized Marker DotPlot for {d_name}...")
            dp = sc.pl.dotplot(
                adata_curr,
                var_names=top_markers_dict,
                groupby=domain_col,
                figsize=(14, 5.5),
                standard_scale='var',
                cmap='Blues',
                title=f"DualGate-DEC: Distinguishing Biomarkers across Spatial Domains ({d_name})",
                show=False,
                return_fig=True
            )

            fig_dot = dp.get_axes()['mainplot_ax'].get_figure()
            save_dot_path = os.path.join(VISUALS_DIR, f"DualGate_DEC_biomarker_dotplot_{d_name}.png")
            fig_dot.savefig(save_dot_path, dpi=300, bbox_inches='tight')
            print(f"   ► [Saved] Standardized dotplot saved to: {save_dot_path}")
            plt.show()
    except Exception as e:
        print(f"   [Notice] DotPlot rendering notice for {d_name}: {e}")

    # -------------------------------------------------------------
    # 2. Standardized Anatomical Layer Concordance Heatmap (Uniform figsize=(13, 7))
    # -------------------------------------------------------------
    try:
        gt_series = adata_curr.obs.get('ground_truth', None)
        if gt_series is not None and gt_series.dropna().nunique() > 1:
            print(f"[2/2] Generating Standardized Layer Concordance Heatmap for {d_name}...")
            annotated_mask = gt_series.notna() & (gt_series != 'nan') & (gt_series != '-1')
            if annotated_mask.sum() > 0:
                ct = pd.crosstab(gt_series[annotated_mask], adata_curr.obs.loc[annotated_mask, domain_col], normalize='index') * 100.0

                # Strictly sort domain columns numerically (Domain 0, 1, 2... instead of alphabetical 0, 1, 10...)
                sorted_cols = sorted(ct.columns, key=lambda x: int(x) if str(x).isdigit() else str(x))
                ct = ct[sorted_cols]
                n_domains = len(sorted_cols)

                # Standardized publication canvas (uniform size across all datasets)
                fig_ct, ax_ct = plt.subplots(figsize=(13, 7))
                im = ax_ct.imshow(ct.values, cmap='YlGnBu', aspect='auto', vmin=0, vmax=100)

                ax_ct.set_xticks(range(len(ct.columns)))
                ax_ct.set_xticklabels([f"Domain {c}" for c in ct.columns], rotation=35, ha='right', fontsize=10)
                ax_ct.set_yticks(range(len(ct.index)))
                ax_ct.set_yticklabels(ct.index, fontsize=10)
                ax_ct.set_xlabel("DualGate-DEC Discovered Domains", fontweight='bold', fontsize=11)
                ax_ct.set_ylabel("Ground Truth Manual Annotations", fontweight='bold', fontsize=11)
                ax_ct.set_title(f"{d_name}\nAnatomical Layer Concordance Heatmap (%)", fontweight='bold', fontsize=12)

                divider = make_axes_locatable(ax_ct)
                cax = divider.append_axes("right", size="3%", pad=0.08)
                cbar = fig_ct.colorbar(im, cax=cax)
                cbar.set_label("Layer Overlap Concordance (%)", fontsize=10)

                for r in range(ct.shape[0]):
                    for c in range(ct.shape[1]):
                        val = ct.values[r, c]
                        if val >= 5.0:
                            ax_ct.text(
                                c, r, f"{val:.1f}%",
                                ha='center', va='center',
                                color='white' if val > 55 else 'black',
                                fontsize=8.5 if n_domains > 10 else 9.5, fontweight='bold'
                            )

                plt.tight_layout()
                save_ct_path = os.path.join(VISUALS_DIR, f"DualGate_DEC_domain_concordance_{d_name}.png")
                plt.savefig(save_ct_path, dpi=300, bbox_inches='tight')
                print(f"   ► [Saved] Standardized layer concordance heatmap saved to: {save_ct_path}")
                plt.show()
    except Exception as e:
        print(f"   [Notice] Layer concordance rendering notice for {d_name}: {e}")

print("\n" + "="*85)
print(" ALL STANDARDIZED BIOMARKER VISUALIZATIONS COMPLETE & SAVED TO visuals/")
print("="*85 + "\n")


In [ ]:
# Cell 16: Upload & Synchronize All Outputs to Google Drive
import os
import shutil
import datetime

# 1. Mount Google Drive if in Google Colab environment
try:
    from google.colab import drive
    print("[Drive Sync] Mounting Google Drive...")
    drive.mount('/content/drive')
    in_colab = True
except Exception as e:
    in_colab = False
    print(f"[Notice] Google Drive mount not available ({e}). Running in local environment.")

# 2. Upload output folder to Google Drive
OUTPUT_DIR = "DualGate_DEC_outputs" if 'OUTPUT_DIR' not in locals() else OUTPUT_DIR

# Set local timezone offset (UTC+6 for Bangladesh Standard Time, or customize as needed)
TIMEZONE_OFFSET_HOURS = 6
local_tz = datetime.timezone(datetime.timedelta(hours=TIMEZONE_OFFSET_HOURS))
now_local = datetime.datetime.now(datetime.timezone.utc).astimezone(local_tz)

# Crystal-clear human-readable date and time (e.g., "30-Sep-2026_04-05-12AM")
timestamp_readable = now_local.strftime("%d-%b-%Y_%I-%M-%S%p")

if in_colab:
    drive_dest_dir = "/content/drive/MyDrive/DualGate_DEC_outputs"
    os.makedirs(drive_dest_dir, exist_ok=True)

    print(f"\n[Drive Sync] Uploading full output folder '{OUTPUT_DIR}' to: '{drive_dest_dir}'...")
    # Copy results, visuals, biomarkers recursively
    shutil.copytree(OUTPUT_DIR, drive_dest_dir, dirs_exist_ok=True)

    # Create a compressed human-readable ZIP archive in MyDrive
    zip_path_no_ext = f"/content/drive/MyDrive/DualGate_DEC_outputs_{timestamp_readable}"
    print(f"[Drive Sync] Creating backup ZIP archive: '{zip_path_no_ext}.zip'...")
    shutil.make_archive(zip_path_no_ext, 'zip', OUTPUT_DIR)

    print("\n" + "="*85)
    print(" [SUCCESS] DUALGATE-DEC OUTPUT FOLDER FULLY SYNCHRONIZED TO GOOGLE DRIVE!")
    print(f" * Synced Folder : {drive_dest_dir}/")
    print(f"   |-- results/   : Benchmark CSVs & processed AnnData .h5ad files")
    print(f"   |-- visuals/   : 300 DPI publication charts, trajectories & UMAPs")
    print(f"   +-- biomarkers/: Full Wilcoxon marker gene CSV tables")
    print(f" * Backup ZIP    : {zip_path_no_ext}.zip")
    print(f" * Timestamp     : {now_local.strftime('%A, %B %d, %Y at %I:%M:%S %p')} (UTC+{TIMEZONE_OFFSET_HOURS})")
    print("="*85 + "\n")
else:
    print(f"\n[Local Environment] All outputs safely stored locally at: '{os.path.abspath(OUTPUT_DIR)}'")
